<a href="https://colab.research.google.com/github/DrewThomasson/ebook2audiobook/blob/main/Notebooks/colab_ebook2audiobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/DrewThomasson/ebook2audiobook)

In [ ]:
# @title 🚀 Run ebook2audiobook!

import os
import re
import subprocess
from collections import deque
from pathlib import Path

REPO_DIR = Path("/content/ebook2audiobook")
REPO_URL = "https://github.com/DrewThomasson/ebook2audiobook.git"
REPO_BRANCH = "main"
LOG_PATH = Path("/content/ebook2audiobook_install.log")
MAX_DISPLAYED_LINES = 1000

if subprocess.run(
    ["nvidia-smi"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
).returncode != 0:
    raise RuntimeError(
        "No NVIDIA GPU detected. Select an NVIDIA GPU accelerator in the "
        "notebook settings, restart the session, and run this cell again."
    )

if not (REPO_DIR / ".git").exists():
    if REPO_DIR.exists():
        raise RuntimeError(
            f"{REPO_DIR} exists but is not a Git checkout. "
            "Restart the notebook session and try again."
        )
    print("Cloning ebook2audiobook...", flush=True)
    subprocess.run(
        ["git", "clone", "--depth=1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    print("Using the existing ebook2audiobook checkout.", flush=True)

# Hosted NVIDIA notebooks use the CUDA 12.8 wheels. Keep this workaround local
# to the temporary notebook checkout until the v26 compatibility matrix catches up.
config_path = REPO_DIR / "lib" / "conf.py"
config_lines = config_path.read_text(encoding="utf-8").splitlines(keepends=True)
for index, line in enumerate(config_lines):
    if '"cu128":' in line:
        config_lines[index] = re.sub(
            r'"last":\s*"2\.13\.0"',
            '"last": "2.11.0"',
            line,
        )
        break
config_path.write_text("".join(config_lines), encoding="utf-8")

env = os.environ.copy()
env.update({
    "DEVICE_TAG": "cu128",
    "MPLBACKEND": "Agg",
    "PYTHONUNBUFFERED": "1",
    "TMPDIR": "/tmp",
})

progress_terms = (
    "creating ./python_env",
    "hardware detected",
    "installing python",
    "installing missing",
    "all required packages",
    "downloaded",
    "running on local",
    "running on public",
    "gradio.live",
)
error_terms = ("error", "failed", "traceback", "exception", "critical", "fatal")

print("Starting ebook2audiobook...", flush=True)
print("The first installation commonly takes 20–60 minutes.", flush=True)
print(f"The complete log is being saved to {LOG_PATH}", flush=True)
print(f"Live notebook output is capped at {MAX_DISPLAYED_LINES} important lines.\n", flush=True)

process = subprocess.Popen(
    ["bash", "./ebook2audiobook.command", "--share"],
    cwd=REPO_DIR,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

displayed_lines = 0
server_started = False
recent_lines = deque(maxlen=40)

try:
    with LOG_PATH.open("w", encoding="utf-8") as log:
        assert process.stdout is not None
        for line in process.stdout:
            log.write(line)
            log.flush()
            recent_lines.append(line)

            line_lower = line.lower()
            if "running on public" in line_lower or "gradio.live" in line_lower:
                server_started = True
            show_line = any(term in line_lower for term in progress_terms)
            show_line = show_line or (
                server_started and any(term in line_lower for term in error_terms)
            )
            if (
                displayed_lines < MAX_DISPLAYED_LINES
                and show_line
            ):
                print(line, end="", flush=True)
                displayed_lines += 1
                if displayed_lines == MAX_DISPLAYED_LINES:
                    print(
                        "\nLive output limit reached. The app is still running; "
                        f"additional output is being saved to {LOG_PATH}",
                        flush=True,
                    )
except KeyboardInterrupt:
    process.terminate()
    raise
finally:
    if process.stdout is not None:
        process.stdout.close()

return_code = process.wait()
if return_code != 0:
    print(f"\nebook2audiobook exited with code {return_code}.", flush=True)
    print("Last 40 log lines:\n", flush=True)
    print("".join(recent_lines), flush=True)
    print(f"Complete log: {LOG_PATH}", flush=True)
    raise RuntimeError("ebook2audiobook failed to start; see the log above.")

print("\nebook2audiobook stopped normally.", flush=True)
